# Feature Extraction (Autoencoder)

Per the supervisor's request, this notebook adds Autoencoder-based feature extraction as an **additional, parallel experimental branch** -- it does not replace the engineered features from Notebook 06, and Notebook 07 (Model Development) continues to run its full original analysis on those engineered features exactly as before.

This notebook was originally built as a section inside Notebook 07 itself, but was split out into its own standalone notebook after the autoencoder cells were found to have been silently skipped during a run of the combined notebook -- everything downstream of them still executed successfully using only the original features, with no error to signal that the encoded branch had not actually run. Splitting this into its own notebook means it can be run and verified independently, with its own clear pass/fail signal, before Notebook 07 ever depends on its output.

**Pipeline position:**

```
Data Collection -> ... -> Feature Engineering (06)
                                |
                                v
                    Feature Extraction (Autoencoder) (this notebook)
                                |
                                v
                       Model Development (07)
```

**What this notebook does:** trains one autoencoder per (horizon, asset, feature configuration) combination -- the same granularity as every model in Notebook 07, since feature composition differs by configuration and the target differs by horizon -- and saves the resulting encoded train/validation/test sets to disk. Notebook 07 loads these saved files directly; it does not retrain or depend on this notebook's in-memory state.

**Leakage discipline**, matching the LSTM pattern already established in Notebook 07: the autoencoder's weights are fit only on training-split data; the validation split is used solely for `EarlyStopping`'s monitored loss (monitoring, not training); the test split is only ever encoded through the already-fitted encoder, never used to fit anything. The `StandardScaler` is fit on the training split alone and applied unchanged to validation and test.

**Scope excluded:** ARIMA and Naive are not part of this notebook at all. ARIMA is univariate on the price series alone and never receives a feature matrix; Naive has no features or parameters. "Encoded features" does not apply to either, structurally.

In [1]:
import sys
import os
import json
from pathlib import Path

# Must be set BEFORE tensorflow is imported (directly below, and
# indirectly via src.models.autoencoder in the next cell) -- confirmed,
# not guessed, to fix a real, reproducible silent freeze during
# model.fit() on this machine (isolated test: froze indefinitely with
# TensorFlow's default multi-threaded settings, completed in under a
# second with these two set). See test_tf_hang.py for the isolated
# reproduction this fix was verified against.
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

pd.set_option("display.max_columns", None)


In [2]:
# same root-finding trick as notebooks 05/06/07
def find_project_root(start, marker):
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"can't find {marker} anywhere above {start}")

PROJECT_ROOT = find_project_root(Path.cwd(), "src")
sys.path.insert(0, str(PROJECT_ROOT))

FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
OUT_DIR = PROJECT_ROOT / "results" / "autoencoder"
AE_MODELS_DIR = OUT_DIR / "models"
AE_DATA_DIR = OUT_DIR / "encoded_features"
AE_METRICS_DIR = OUT_DIR / "metrics"
for d in [AE_MODELS_DIR, AE_DATA_DIR, AE_METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Reading feature sets from: {FEATURES_DIR}")
print(f"Saving autoencoder outputs to: {OUT_DIR}")

# Reusable autoencoder architecture, training, encoding, and save/load logic
# now lives in src/models/autoencoder.py, imported here rather than defined
# inline -- matching the pattern already used for feature engineering
# (Notebook 06) and modelling (Notebook 07). This notebook remains the place
# where every parameter, training run, reconstruction loss, and saved
# artefact is actually visible and printed; src/ is only the reusable
# mechanism, and is also what a future inference pipeline would import.
from src.models.autoencoder import (
    train_autoencoder, encode, reconstruction_loss, save_autoencoder,
)

Project root: /Users/mas364/Desktop/AI_Crypto_Forecasting
Reading feature sets from: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features
Saving autoencoder outputs to: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/autoencoder


In [3]:
ASSETS = {"Bitcoin": "Bitcoin_Price", "Ethereum": "Ethereum_Price"}
FORECAST_HORIZONS = [1, 7, 14]

def horizon_suffix_for(horizon):
    return f"_{horizon}day" if horizon != 1 else ""

FEATURE_FILE_TEMPLATES = {
    "primary": "{a}_market_only_features{s}.csv",
    "candidate": "{a}_market_google_trends_features{s}.csv",
    "ablation": "{a}_market_fear_greed_features{s}.csv",
    "combined": "{a}_market_all_external_features{s}.csv",
}
DISPLAY_NAMES = {
    "primary": "Market Only",
    "candidate": "Market + Google Trends",
    "ablation": "Market + Fear & Greed Index",
    "combined": "Market + Google Trends + Fear & Greed Index",
}

# Matching Notebook 07's split exactly -- same fractions, same chronological,
# no-shuffle logic, so the train/val/test rows encoded here are identical to
# what Notebook 07 itself would split them into.
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

ENCODING_DIM = 8
AE_EPOCHS = 150
AE_PATIENCE = 15
AE_SEED = 42

print(f"Autoencoder config: encoding_dim={ENCODING_DIM}, max_epochs={AE_EPOCHS}, patience={AE_PATIENCE}")

Autoencoder config: encoding_dim=8, max_epochs=150, patience=15


## Loading Feature Sets and Splitting

Identical logic to Notebook 07 -- same feature file loading, same chronological 70/15/15 split with no shuffling, same assertions confirming no temporal overlap between partitions. This is not a coincidence: it guarantees the train/val/test rows encoded in this notebook line up exactly with the rows Notebook 07 itself works with.

In [4]:
def load_feature_set(asset, fset, price_col, horizon):
    fname = FEATURE_FILE_TEMPLATES[fset].format(a=asset.lower(), s=horizon_suffix_for(horizon))
    path = FEATURES_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run 06_feature_engineering.ipynb first, for horizon={horizon}."
        )
    df = pd.read_csv(path, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
    target_col = f"{price_col}_log_return_target_t_plus_{horizon}"
    assert target_col in df.columns
    assert df["Date"].is_monotonic_increasing
    assert not df["Date"].duplicated().any()
    return df


def split_data(df):
    n = len(df)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

    train = df.iloc[:train_end].reset_index(drop=True)
    val = df.iloc[train_end:val_end].reset_index(drop=True)
    test = df.iloc[val_end:].reset_index(drop=True)

    assert train["Date"].max() < val["Date"].min()
    assert val["Date"].max() < test["Date"].min()

    return train, val, test


def get_X_y(df, price_col, horizon):
    target_col = f"{price_col}_log_return_target_t_plus_{horizon}"
    level_col = f"{price_col}_level"
    X = df.drop(columns=["Date", level_col, target_col])
    y = df[target_col]
    return X, y


data, splits = {}, {}
for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            df = load_feature_set(asset, fset, price_col, horizon)
            data[(horizon, asset, fset)] = df
            splits[(horizon, asset, fset)] = split_data(df)

print(f"Loaded and split {len(data)} feature sets across {len(FORECAST_HORIZONS)} horizons.")

Loaded and split 24 feature sets across 3 horizons.


## Autoencoder Architecture and Training

A simple symmetric dense autoencoder: input -> 16 -> bottleneck (8) -> 16 -> input, trained to reconstruct its own scaled input.

**Why 8.** The original feature counts range from 14 (Market Only) to roughly 27 (the combined configuration) -- 8 keeps the bottleneck well below even the smallest input size, forcing genuine compression rather than a near-identity mapping, while staying large enough to plausibly retain the small number of features (own-asset lags, rolling statistics) already known from Notebook 05 to carry most of the real signal. This is a deliberately simple, round-number choice rather than a tuned hyperparameter -- stated as a scope decision, not hidden, since tuning it per configuration was judged to add complexity disproportionate to its value for what remains an exploratory branch of this project. `EarlyStopping` monitors validation reconstruction loss -- validation data is never used to update the model's weights, only to decide when to stop training, exactly as `EarlyStopping` is already used for LSTM in Notebook 07.

The encoding dimension (8) is fixed for every configuration rather than tuned per configuration -- a deliberate scope decision for what is an additional exploratory branch, not the project's primary modelling approach, stated here rather than left implicit.

In [5]:
# build_autoencoder() and train_autoencoder() are imported above from
# src.models.autoencoder -- see the imports cell and the markdown above
# for the architecture and leakage-discipline explanation.
print("Autoencoder architecture and training functions imported from src.models.autoencoder")

Autoencoder architecture and training functions imported from src.models.autoencoder


In [6]:
# DIAGNOSTIC CELL v3 -- run this INSTEAD of the previous diagnostic cell.
# Checks per-column variance and correlation on the exact real data that
# froze, WITHOUT touching TensorFlow at all -- pure pandas/numpy, so this
# cannot hang regardless of what's actually wrong.

horizon, asset, fset = 1, "Bitcoin", "primary"
train, val, test = splits[(horizon, asset, fset)]
price_col = ASSETS[asset]
X_train_real, y_train_real = get_X_y(train, price_col, horizon)
X50 = X_train_real.head(50)

print("Per-column standard deviation (first 50 real rows):")
stds = X50.std()
print(stds.sort_values().to_string())
print()

print("Columns with near-zero variance (std < 0.0001):")
near_zero = stds[stds < 0.0001]
print(near_zero if len(near_zero) else "  none")
print()

print("Highly correlated column pairs (|correlation| > 0.999, excluding self-pairs):")
import numpy as np
corr = np.array(X50.corr().abs().values, copy=True)
np.fill_diagonal(corr, 0)
corr_df = pd.DataFrame(corr, index=X50.columns, columns=X50.columns)
high_corr = corr_df.stack()
high_corr = high_corr[high_corr > 0.999].sort_values(ascending=False)
print(high_corr.head(20) if len(high_corr) else "  none")

Per-column standard deviation (first 50 real rows):
Bitcoin_Price_log_return_rolling_mean_30    6.558176e-03
Bitcoin_Price_log_return_rolling_std_30     7.123000e-03
Bitcoin_Price_log_return                    3.112058e-02
Bitcoin_Price_log_return_lag_2              3.119251e-02
Bitcoin_Price_log_return_lag_1              3.129273e-02
Bitcoin_Price_log_return_lag_7              3.474954e-02
Bitcoin_Volume_pct_change_lag_1             2.652600e-01
Bitcoin_Volume_pct_change_lag_7             2.678106e-01
Bitcoin_Volume_log                          2.738944e-01
Bitcoin_Price_momentum_7                    3.349865e+03
Bitcoin_Volume_rolling_std_30               1.010303e+09
Bitcoin_Volume_rolling_mean_30              3.942586e+09
Bitcoin_Volume_raw_lag_1                    8.075550e+09
Bitcoin_Volume_raw_lag_7                    8.106699e+09

Columns with near-zero variance (std < 0.0001):
  none

Highly correlated column pairs (|correlation| > 0.999, excluding self-pairs):
  none


## Training, Encoding, and Saving -- All 24 Configurations

One autoencoder per (horizon, asset, feature configuration) combination. For each: fit on training data, encode all three splits through the fitted encoder, save the encoder, autoencoder, scaler, encoded CSVs, training history, and reconstruction loss.

Resumable -- a combination already saved on disk is skipped rather than retrained, so re-running this cell after an interruption only repeats whatever genuinely did not finish.


In [7]:
autoencoder_manifest_rows = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}"
            label = f"h={horizon:2d}d {asset:9s} {DISPLAY_NAMES[fset]:38s}"

            # RESUMABLE: metadata.json is the last file save_autoencoder()
            # writes for a given combination, so its presence means every
            # earlier file (both .keras models, the scaler, the encoded
            # CSVs) was already written successfully too. If a previous
            # run hung or crashed partway through the 24 combinations,
            # re-running this cell only redoes whatever is genuinely
            # unfinished, instead of retraining from combination 1 again.
            metadata_path = AE_MODELS_DIR / f"{prefix}_metadata.json"
            if metadata_path.exists():
                saved_meta = json.load(open(metadata_path))
                autoencoder_manifest_rows.append({
                    "horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset],
                    "original_feature_count": saved_meta["input_dim"], "encoding_dim": saved_meta["encoding_dim"],
                    "epochs_trained": saved_meta["epochs_trained"],
                    "reconstruction_loss_train": saved_meta.get("reconstruction_loss_train"),
                    "reconstruction_loss_val": saved_meta.get("reconstruction_loss_val"),
                    "reconstruction_loss_test": saved_meta.get("reconstruction_loss_test"),
                })
                print(f"{label} SKIPPED -- already saved ({metadata_path.name} found)")
                continue

            # Printed BEFORE training starts, not just after it finishes --
            # if this cell hangs or crashes, the last line printed tells
            # you exactly which combination it was on, rather than leaving
            # you with no information at all (the previous version of this
            # cell only printed once a combination had fully completed).
            print(f"{label} starting...")

            train, val, test = splits[(horizon, asset, fset)]
            level_col = f"{price_col}_level"
            target_col = f"{price_col}_log_return_target_t_plus_{horizon}"

            X_train, y_train = get_X_y(train, price_col, horizon)
            X_val, y_val = get_X_y(val, price_col, horizon)
            X_test, y_test = get_X_y(test, price_col, horizon)

            autoencoder, encoder, scaler, history = train_autoencoder(
                X_train, X_val, encoding_dim=ENCODING_DIM, epochs=AE_EPOCHS, patience=AE_PATIENCE, seed=AE_SEED,
            )

            encoded_train = encode(encoder, scaler, X_train)
            encoded_val = encode(encoder, scaler, X_val)
            encoded_test = encode(encoder, scaler, X_test)

            recon_train = reconstruction_loss(autoencoder, scaler, X_train)
            recon_val = reconstruction_loss(autoencoder, scaler, X_val)
            recon_test = reconstruction_loss(autoencoder, scaler, X_test)

            encoded_cols = [f"encoded_{i}" for i in range(ENCODING_DIM)]

            def _rebuild(encoded_array, source_df):
                edf = pd.DataFrame(encoded_array, columns=encoded_cols)
                edf["Date"] = source_df["Date"].values
                edf[level_col] = source_df[level_col].values
                edf[target_col] = source_df[target_col].values
                return edf[["Date"] + encoded_cols + [level_col, target_col]]

            _rebuild(encoded_train, train).to_csv(AE_DATA_DIR / f"{prefix}_encoded_train.csv", index=False)
            _rebuild(encoded_val, val).to_csv(AE_DATA_DIR / f"{prefix}_encoded_validation.csv", index=False)
            _rebuild(encoded_test, test).to_csv(AE_DATA_DIR / f"{prefix}_encoded_test.csv", index=False)

            # save_autoencoder() writes the same _autoencoder.keras / _encoder.keras /
            # _scaler.joblib / _training_history.csv files as before, plus a new
            # _metadata.json (encoding_dim, epochs trained, reconstruction loss,
            # horizon/asset/feature_set) that did not exist before this module existed --
            # this is what a future inference pipeline reads to validate its inputs, and
            # what the resumability check above reads to skip already-finished work.
            saved_paths = save_autoencoder(
                autoencoder, encoder, scaler, history, AE_MODELS_DIR / prefix,
                metadata={
                    "horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset],
                    "reconstruction_loss_train": recon_train, "reconstruction_loss_val": recon_val,
                    "reconstruction_loss_test": recon_test,
                },
            )
            # history CSV is also written by save_autoencoder(), under AE_MODELS_DIR --
            # kept here too under AE_METRICS_DIR for backward-compatible file location.
            pd.DataFrame(history.history).to_csv(AE_METRICS_DIR / f"{prefix}_training_history.csv", index=False)

            n_epochs_trained = len(history.history["loss"])
            autoencoder_manifest_rows.append({
                "horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset],
                "original_feature_count": X_train.shape[1], "encoding_dim": ENCODING_DIM,
                "epochs_trained": n_epochs_trained,
                "reconstruction_loss_train": recon_train, "reconstruction_loss_val": recon_val,
                "reconstruction_loss_test": recon_test,
            })

            print(f"{label} "
                  f"{X_train.shape[1]:2d}->{ENCODING_DIM} features | "
                  f"epochs={n_epochs_trained:3d} | test recon loss={recon_test:.4f}")

autoencoder_manifest = pd.DataFrame(autoencoder_manifest_rows)
manifest_save_path = AE_METRICS_DIR / "autoencoder_manifest.csv"
autoencoder_manifest.to_csv(manifest_save_path, index=False)
print(f"\nSaved manifest: {manifest_save_path}")
display(autoencoder_manifest.round(4))

h= 1d Bitcoin   Market Only                            SKIPPED -- already saved (bitcoin_primary_metadata.json found)
h= 1d Bitcoin   Market + Google Trends                 SKIPPED -- already saved (bitcoin_candidate_metadata.json found)
h= 1d Bitcoin   Market + Fear & Greed Index            SKIPPED -- already saved (bitcoin_ablation_metadata.json found)
h= 1d Bitcoin   Market + Google Trends + Fear & Greed Index SKIPPED -- already saved (bitcoin_combined_metadata.json found)
h= 1d Ethereum  Market Only                            SKIPPED -- already saved (ethereum_primary_metadata.json found)
h= 1d Ethereum  Market + Google Trends                 SKIPPED -- already saved (ethereum_candidate_metadata.json found)
h= 1d Ethereum  Market + Fear & Greed Index            SKIPPED -- already saved (ethereum_ablation_metadata.json found)
h= 1d Ethereum  Market + Google Trends + Fear & Greed Index SKIPPED -- already saved (ethereum_combined_metadata.json found)
h= 7d Bitcoin   Market Only       

,horizon,asset,feature_set,original_feature_count,encoding_dim,epochs_trained,reconstruction_loss_train,reconstruction_loss_val,reconstruction_loss_test
0,1,Bitcoin,Market Only,14,8,150,0.1020,0.3495,0.4016
1,1,Bitcoin,Market + Google Trends,18,8,106,0.2317,0.3699,0.4120
2,1,Bitcoin,Market + Fear & Greed Index,20,8,99,0.1854,0.3403,0.3990
3,1,Bitcoin,Market + Google Trends + Fear & Greed Index,24,8,150,0.2260,0.4059,0.4680
4,1,Ethereum,Market Only,14,8,150,0.1456,0.4842,0.6246
5,1,Ethereum,Market + Google Trends,18,8,112,0.2139,0.4285,0.6631
6,1,Ethereum,Market + Fear & Greed Index,20,8,150,0.1680,0.4516,0.7368
7,1,Ethereum,Market + Google Trends + Fear & Greed Index,24,8,150,0.2247,0.4912,0.7775
8,7,Bitcoin,Market Only,15,8,150,0.1047,0.3691,0.3672
9,7,Bitcoin,Market + Google Trends,19,8,59,0.2751,0.3580,0.4062


## Verification

Confirms every expected output file exists, encoded feature counts match `ENCODING_DIM`, and there is no leakage: the saved scaler and encoder were fit on training data only (verified structurally above, by construction -- `train_autoencoder` never receives test data as an argument), and every encoded file's row count matches its corresponding original split exactly.

In [8]:
expected_files = 0
missing_files = []

for horizon in FORECAST_HORIZONS:
    for asset in ASSETS:
        for fset in FEATURE_FILE_TEMPLATES:
            prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}"
            for suffix in ["_autoencoder.keras", "_encoder.keras", "_scaler.joblib", "_metadata.json",
                           "_encoded_train.csv", "_encoded_validation.csv", "_encoded_test.csv"]:
                expected_files += 1
                path = (AE_MODELS_DIR if "keras" in suffix or "joblib" in suffix or "json" in suffix else AE_DATA_DIR) / f"{prefix}{suffix}"
                if not path.exists():
                    missing_files.append(str(path))

print(f"Expected files: {expected_files}")
print(f"Missing files: {len(missing_files)}")
if missing_files:
    for f in missing_files:
        print("  MISSING:", f)
assert not missing_files, "Some expected autoencoder outputs were not saved -- see above."

# Row-count consistency: encoded splits must match their original splits exactly
mismatches = []
for horizon in FORECAST_HORIZONS:
    for asset in ASSETS:
        for fset in FEATURE_FILE_TEMPLATES:
            train, val, test = splits[(horizon, asset, fset)]
            prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}"
            for split_name, original_split in [("train", train), ("validation", val), ("test", test)]:
                encoded_df = pd.read_csv(AE_DATA_DIR / f"{prefix}_encoded_{split_name}.csv")
                if len(encoded_df) != len(original_split):
                    mismatches.append((prefix, split_name, len(encoded_df), len(original_split)))
                if encoded_df.shape[1] != ENCODING_DIM + 3:  # encoded cols + Date + level + target
                    mismatches.append((prefix, split_name, "column count", encoded_df.shape[1]))

print(f"\nRow/column count mismatches between encoded and original splits: {len(mismatches)}")
for m in mismatches:
    print("  MISMATCH:", m)
assert not mismatches, "Encoded splits do not match original splits in row or column count."

print("\nAll autoencoder outputs verified: files present, row counts match originals, no leakage by construction.")

Expected files: 168
Missing files: 0

Row/column count mismatches between encoded and original splits: 0

All autoencoder outputs verified: files present, row counts match originals, no leakage by construction.


## Summary

Trained 24 autoencoders (3 horizons x 2 assets x 4 feature configurations), each compressing that configuration's engineered features down to 8 encoded dimensions. All encoders were fit on training data only; validation data was used exclusively for early stopping; test data was only ever encoded, never used to fit anything.

**What Notebook 07 loads from this notebook:** the encoded CSVs in `results/autoencoder/encoded_features/` -- one train/validation/test file per (horizon, asset, feature configuration) combination, with the same `Date`, price-level, and target columns as the original feature files, so they can be used identically by `run_random_forest`, `run_xgboost`, and `run_lstm` with no special-casing required.

Random Forest, XGBoost, and LSTM are then trained on these encoded features in Notebook 07, tagged `representation = "Autoencoder-Encoded"`, directly alongside the original `representation = "Original"` results already in that notebook -- both branches in one combined comparison, exactly as the supervisor requested.